# CTB ProSiT reproduction (Google Colab)
Upload this complete folder to `MyDrive/CTB_ProSiT_reproduction`, open this notebook in Colab, and select **Run all**.

## 1. Mount Drive and create an isolated Python environment
The isolated environment prevents Colab's preinstalled NumPy and pandas versions from changing the run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        'Select a Python 3.11 Colab runtime and run the notebook again.'
    )

ROOT = Path('/content/drive/MyDrive/CTB_ProSiT_reproduction')
if not (ROOT / 'reproduce.py').is_file():
    raise FileNotFoundError(f'Handover folder not found: {ROOT}')

VENV = Path('/content/ctb_prosit_env')
PYTHON = VENV / 'bin' / 'python'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], check=True)
if not PYTHON.is_file():
    subprocess.run([sys.executable, '-m', 'virtualenv', str(VENV)], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)

ENV = os.environ.copy()
ENV['MPLBACKEND'] = 'Agg'
ENV['PYTHONUNBUFFERED'] = '1'

def run_isolated(source):
    subprocess.run([str(PYTHON), '-c', source], cwd=ROOT, env=ENV, check=True)

print('Isolated environment:', PYTHON)

## 2. Load and inspect PNML, JSON and PKL models
JSON is the readable ProSiT export; PKL preserves the exact calibrated runtime object used for the thesis.

In [ ]:
run_isolated("""
import reproduce
print('JSON models loaded with SimulatorParameters.from_json():')
print(reproduce.inspect_models(reproduce.load_json_models()).to_string(index=False))
models = reproduce.load_pickle_models()
print('\nExact thesis PKL models:')
print(reproduce.inspect_models(models).to_string(index=False))
print('\nIntervention checks:')
print(reproduce.check_model_changes(models).to_string(index=False))
""")

## 3. Reproduce and compare the thesis scenario results
The default executes 10 matched seeds × 3 models × 17,892 cases and can take approximately 30 minutes.

In [ ]:
run_isolated("""
import reproduce
RUN_FULL = True
output_dir = reproduce.run(full=RUN_FULL)
print(f'Fresh result tables: {output_dir}')
if RUN_FULL:
    comparison = reproduce.compare(output_dir)
    print(comparison.to_string(index=False))
    print('FULL REPRODUCTION PASSED')
""")

## 4. Run the demand-saturation diagnostic
This derives six arrival-intensity levels from the saved baseline and tests the simulator around the static saturation estimate.

In [ ]:
run_isolated("""
import saturation_experiment
RUN_FULL = True
output_dir = saturation_experiment.run(full=RUN_FULL)
print(f'Fresh saturation tables: {output_dir}')
if RUN_FULL:
    comparison = saturation_experiment.compare(output_dir)
    print(comparison.to_string(index=False))
    print('SATURATION REPRODUCTION PASSED')
""")

## Reading the results
`outputs/full/scenario_paired_delta_summary.csv` contains the paired policy effects. `outputs/saturation_full/` contains the saturation response curve and paired effects. The first strong response occurs at a realised elapsed-rate ratio of 2.367: mean turnaround increases by 4.644 minutes and mean RMG pre-service by 5.878 minutes. Each comparison file is the technical reproducibility check. The notebook reproduces simulation from frozen models, not discovery from the confidential CTB event log.